## Transform Customer Data
### 1.Remove the Records with Null Customer_id
### 2.Reomove Extact duplicates Records
### 3.Remove the Duplicate based on created Timestamps
### 4.Cast the correct data Types
### 5.Transform to silver Schema

# Remove the records with Null Customer_id

In [0]:
df = spark.sql('SELECT * FROM gizmobox_nara.bronze.py_customers')

USING PYTHON

In [0]:
df = spark.read.table('gizmobox_nara.bronze.py_customers')
display(df)

In [0]:
df_filter = df.filter('customer_id IS NOT NULL')
display(df_filter)

### using object notation

In [0]:
df_filter = df.where(df.customer_id.isNotNull())
display(df_filter)

# Remove the Duplicates

In [0]:
df_distinct = df_filter.distinct()
display(df_distinct)

In [0]:
df_distinct = df_filter.dropDuplicates()
display(df_distinct)

In [0]:
from pyspark.sql import functions as F
df_max = df_distinct.groupBy('customer_id') \
         .agg(F.max('created_timestamp').alias('max_created_timestamp'))
display(df_max)

In [0]:
%sql
SELECT customer_id,
      MAX(created_timestamp) AS max_created_timestamp
      FROM v_customers_distinct
      GROUP BY customer_id

NEW DATA FRAME JOINING DISTINCT 

In [0]:
df_distinct_customer = (
    df_distinct
    .join(df_max, (df_distinct.customer_id == df_max.customer_id) & (df_distinct.created_timestamp == df_max.max_created_timestamp), 'inner')
    .select(df_distinct['*'])
)
display(df_distinct_customer)

### CAST the column to correct data type

In [0]:
df_casted_customer =(
                     df_distinct_customer
                    .select(df_distinct_customer.created_timestamp.cast('timestamp'),
                            df_distinct_customer.customer_id.cast('integer'),
                            df_distinct_customer.customer_name,
                            df_distinct_customer.date_of_birth.cast('date'),
                            df_distinct_customer.email,
                            df_distinct_customer.member_since.cast('date'),
                            df_distinct_customer.telephone)
                    )
display(df_casted_customer)



### Write Data to a Delta Table


In [0]:
df_casted_customer.writeTo('gizmobox_nara.silver.py_customers').createOrReplace()

In [0]:
df= spark.read.table("gizmobox_nara.silver.py_customers")
display(df)

In [0]:
%sql
SELECT * FROM gizmobox_nara.silver.py_customers

In [0]:
%sql
WITH cte_max AS
(
  SELECT customer_id,
      MAX(created_timestamp) AS max_created_timestamp
      FROM v_customers_distinct
      GROUP BY customer_id
)
SELECT CAST(t.created_timestamp AS TIMESTAMP) As created_timestamp,
    t.customer_id,
    t.customer_name,
    CAST(t.date_of_birth AS DATE) AS date_of_birth,
    t.email,
    CAST(t.member_since AS DATE) AS member_since,
    t.telephone
    FROM v_customers_distinct t
    INNER JOIN cte_max m
    ON t.customer_id = m.customer_id
    AND t.created_timestamp = m.max_created_timestamp

In [0]:
%sql
DROP TABLE gizmobox_nara.silver.customers;
CREATE TABLE gizmobox_nara.silver.customers AS
WITH cte_max AS
(
  SELECT customer_id,
      MAX(created_timestamp) AS max_created_timestamp
      FROM v_customers_distinct
      GROUP BY customer_id
)
SELECT CAST(t.created_timestamp AS TIMESTAMP) As created_timestamp,
    t.customer_id,
    t.customer_name,
    CAST(t.date_of_birth AS DATE) AS date_of_birth,
    t.email,
    CAST(t.member_since AS DATE) AS member_since,
    t.telephone
    FROM v_customers_distinct t
    INNER JOIN cte_max m
    ON t.customer_id = m.customer_id
    AND t.created_timestamp = m.max_created_timestamp

In [0]:
%sql
SELECT * FROM gizmobox_nara.silver.customers